# Introduction to AI: E-commerce A/B Personalization
This notebook complements the lecture slides in `slides/introduction_to_ai/slides.md`.

## Story
An online shop tests **UI variant A vs B**. Some customer cohorts convert better with A, others with B.
We first **observe conversion differences** by cohort. Then we train a model to **predict which variant to show** for a new customer.

## Columns (explained simply)
- `customer_id`: unique ID of the customer
- `age`: customer age in years
- `sessions_last30`: number of website visits in the last 30 days
- `avg_basket`: average basket size in EUR (approx.)
- `utm_source`: where the customer came from
  - `organic` = search engine or direct (no paid ad)
  - `paid` = paid ads (Google/Meta ads)
  - `social` = social media posts
  - `email` = newsletter or email campaign
  - `referral` = link from another website
- `ui_variant`: the UI shown to the customer (A or B)
- `converted`: did the customer buy? (1 = yes, 0 = no)
- `cohort`: hidden group used to simulate different behaviors (analysis only)
- `best_variant`: which UI variant is best for that cohort (A or B) — this is our target

**Goal:** predict `best_variant` for new customers.

## Exercises
- Exercise 1: Inspect the dataset and understand features/label.
- Exercise 2: Compare conversion rates for A vs B in each cohort.
- Exercise 3: Train a classifier to predict `best_variant` and evaluate it.

Notes:
- Data generation and train/validation split are already provided.
- `predict_df` is the unlabeled data for your predictions.
- Work through the notebook from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

In [ ]:
# Synthetic e-commerce data (already prepared for you)
rng = np.random.default_rng(42)
n = 1200

segments = np.array(["browsers", "loyalists", "big_spenders", "deal_seekers"])
segment = rng.choice(segments, size=n, p=[0.35, 0.25, 0.2, 0.2])

age = np.empty(n)
sessions_last30 = np.empty(n)
avg_basket = np.empty(n)

# Segment profiles to make clusters clearer
for seg, age_mu, age_sd, sess_mu, sess_sd, basket_mu, basket_sd in [
    ("browsers", 26, 6, 1.5, 0.8, 30, 10),
    ("loyalists", 40, 8, 8.0, 2.0, 90, 20),
    ("big_spenders", 45, 10, 3.0, 1.2, 180, 30),
    ("deal_seekers", 30, 7, 6.0, 1.5, 55, 15),
]:
    mask = segment == seg
    age[mask] = rng.normal(age_mu, age_sd, mask.sum()).clip(18, 70)
    sessions_last30[mask] = rng.normal(sess_mu, sess_sd, mask.sum()).clip(0, 20)
    avg_basket[mask] = rng.normal(basket_mu, basket_sd, mask.sum()).clip(10, 300)

sessions_last30 = np.round(sessions_last30).astype(int)
avg_basket = np.round(avg_basket, 1)

utm_source = np.empty(n, dtype=object)
for seg, probs in [
    ("browsers", [0.4, 0.2, 0.25, 0.05, 0.1]),
    ("loyalists", [0.2, 0.2, 0.1, 0.4, 0.1]),
    ("big_spenders", [0.15, 0.35, 0.05, 0.35, 0.1]),
    ("deal_seekers", [0.3, 0.3, 0.2, 0.05, 0.15]),
]:
    mask = segment == seg
    utm_source[mask] = rng.choice(
        ["organic", "paid", "social", "email", "referral"],
        size=mask.sum(),
        p=probs,
    )

ui_variant = rng.choice(["A", "B"], size=n, p=[0.5, 0.5])

df_raw = pd.DataFrame(
    {
        "customer_id": np.arange(1, n + 1),
        "age": np.round(age, 0).astype(int),
        "sessions_last30": sessions_last30,
        "avg_basket": avg_basket,
        "utm_source": utm_source,
        "ui_variant": ui_variant,
    }
)

# Create cohort and "best variant" labels (for teaching)
df_raw["cohort"] = segment

df_raw["best_variant"] = np.where(
    df_raw["cohort"].isin(["browsers", "deal_seekers"]),
    "A",
    "B",
)

# Create conversion probability (simple, interpretable signal)
# Stronger separation so the model can learn more easily.
base = np.select(
    [
        df_raw["cohort"] == "browsers",
        df_raw["cohort"] == "deal_seekers",
        df_raw["cohort"] == "loyalists",
        df_raw["cohort"] == "big_spenders",
    ],
    [-2.8, -1.6, -0.6, 0.6],
    default=-1.8,
)

variant_effect = np.where(df_raw["ui_variant"] == df_raw["best_variant"], 0.9, -0.3)

logit = base
logit += 0.12 * (sessions_last30 - 3)
logit += 0.015 * (avg_basket - 60)
logit += np.where(utm_source == "email", 0.8, 0.0)
logit += np.where(utm_source == "paid", 0.35, 0.0)
logit += np.where(utm_source == "social", -0.25, 0.0)
logit += variant_effect
logit += rng.normal(0, 0.45, size=n)

prob = 1 / (1 + np.exp(-logit))
df_raw["converted"] = rng.binomial(1, prob)

# Train/validation split (students should not change this)
train_df, valid_df = train_test_split(
    df_raw,
    test_size=0.25,
    random_state=42,
    stratify=df_raw["converted"],
)

# Unlabeled data for predictions (competition)
predict_df = valid_df.drop(columns=["converted", "best_variant", "cohort", "ui_variant"]).copy()
y_valid = valid_df["best_variant"].copy()


def score_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.3f}")
    return acc


def score_on_validation(y_pred):
    return score_predictions(y_valid, y_pred)


df_raw.head()

In [ ]:
# Data prep (given)

df = df_raw.copy()

display(df.head())
display(df.isna().sum())
print(df.shape)
print("Overall conversion rate:", round(df["converted"].mean(), 3))

numeric_cols = ["age", "sessions_last30", "avg_basket"]
categorical_cols = ["utm_source"]
feature_cols = numeric_cols + categorical_cols

TARGET_COL = "best_variant"

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].copy()
X_valid = valid_df[feature_cols].copy()
y_valid = valid_df[TARGET_COL].copy()

print("Features:", feature_cols)
print("Target:", TARGET_COL)

display(X_train.describe())
y_train.value_counts()

## Data overview (already prepared)
- **Rows**: customers
- **Features (input for the model)**: `age`, `sessions_last30`, `avg_basket`, `utm_source`
- **Extra columns (not used in the model)**: `ui_variant`, `cohort`, `converted`
- **Target (label to predict)**: `best_variant` (A or B)

We already prepared:
- `train_df` and `valid_df` (train/validation split)
- `predict_df` (features only, no labels)
- helper function `score_on_validation(y_pred)` to check accuracy

The feature columns and `X_train`, `y_train`, `X_valid`, `y_valid` are defined in the data prep cell above.

## Part A: A/B test analysis by cohort
We check whether **variant A or B** works better for different cohorts.

### Tasks
1. Create a conversion table: cohort × ui_variant.
2. Identify which variant is better for each cohort.
3. Explain in 2-3 simple sentences.

In [ ]:
# Exercise 2: Cohort × Variant conversion table

conversion_table = (
    df.groupby(["cohort", "ui_variant"])["converted"]
    .mean()
    .unstack()
)

conversion_table

### Interpretation (example)
Typical pattern you should see:
- **Browsers / Deal seekers:** Variant **A** converts better.
- **Loyalists / Big spenders:** Variant **B** converts better.

This is exactly why we want a model: choose A or B based on who the customer is.

In [ ]:
# Optional: visualize conversion rates
conversion_table.plot(kind="bar", figsize=(7, 4))
plt.ylabel("Conversion rate")
plt.title("A/B conversion by cohort")
plt.show()

## Part B: Supervised learning - classification
Now we train a model to **predict the best UI variant** for a new customer.
Train/validation sets are already defined for you.

### Tasks
1. One-hot encode categorical features.
2. Build a pipeline with scaling + a classifier.
3. Train, predict, and evaluate.

### Supervised learning: what we are doing (very simple)
- **Input (features X):** age, sessions_last30, avg_basket, utm_source
- **Output (target y):** best_variant (A or B)
- We turn text columns into numbers with **one-hot encoding**.
- The model learns a rule that maps features → best variant.

In practice: fit on `X_train`, then evaluate on `X_valid` using `score_on_validation`.

In [ ]:
# Exercise 3: Classification

X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_valid_enc = pd.get_dummies(X_valid, columns=categorical_cols, drop_first=True)

X_train_enc, X_valid_enc = X_train_enc.align(
    X_valid_enc, join="left", axis=1, fill_value=0
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=300)),
])

clf.fit(X_train_enc, y_train)
y_pred = clf.predict(X_valid_enc)

score_on_validation(y_pred)
print(classification_report(y_valid, y_pred))

In [ ]:
# Plot the confusion matrix for your classifier

ConfusionMatrixDisplay.from_estimator(clf, X_valid_enc, y_valid, cmap="Blues")
plt.title("Confusion matrix")
plt.show()

In [ ]:
# Simple interpretation helpers

baseline_rate = (y_valid == "A").mean()
print("Baseline (most common class) rate:", round(max(baseline_rate, 1 - baseline_rate), 3))

display(
    valid_df.assign(best_variant=y_valid)
    .groupby("best_variant")
    .size()
    .rename("count")
)

### Interpretation (supervised learning)
- **Accuracy vs baseline:** compare your accuracy to the most common class (A or B). Higher is better.
- **Confusion matrix:** shows how many A vs B predictions were correct.
- **Goal:** use the model to choose A or B for a new customer based on their features.

## Discussion and extensions
- Try different values of k and justify your choice.
- Compare clusters to `converted` and discuss mismatches.
- Swap the classifier (DecisionTree, RandomForest, SVM) and compare metrics.
- Compare performance with and without feature scaling.
- Which UI variant performs better overall? Does it depend on device or source?